In [1]:
import xml.etree.ElementTree as ET
from xml.etree.ElementTree import Element
import datetime
import os
import json
import csv


tree = ET.parse("RESULTS_IN_XML_FORMAT.lef")
tree = ET.parse("RESULTS_IN_XML_FORMAT (2).lef")
root = tree.getroot()

In [2]:
elements_keys = {}

for el in root.iter():
    if el.keys():
        elements_keys[el.tag] = el.keys()

elements_keys

{'LENEX': ['version'],
 'CONSTRUCTOR': ['name', 'version'],
 'CONTACT': ['name', 'zip', 'city', 'email', 'internet'],
 'MEET': ['name', 'city', 'nation', 'course', 'timing'],
 'POOL': ['lanemax', 'lanemin'],
 'POINTTABLE': ['name', 'version'],
 'SESSION': ['number', 'date', 'daytime'],
 'EVENT': ['eventid', 'number', 'preveventid', 'gender', 'round', 'daytime'],
 'SWIMSTYLE': ['distance', 'relaycount', 'stroke'],
 'HEAT': ['daytime', 'heatid', 'number'],
 'CLUB': ['name', 'shortname', 'code', 'nation', 'type'],
 'ATHLETE': ['athleteid', 'lastname', 'firstname', 'gender', 'birthdate'],
 'ENTRY': ['entrytime', 'eventid'],
 'RESULT': ['eventid', 'place', 'lane', 'heat', 'swimtime', 'reactiontime'],
 'SPLIT': ['distance', 'swimtime'],
 'RELAY': ['number',
  'agemax',
  'agemin',
  'agetotalmax',
  'agetotalmin',
  'gender',
  'name'],
 'RELAYPOSITION': ['number', 'athleteid', 'reactiontime']}

In [3]:
elements_set = set()

for el in root.iter():
    if not el.keys():
        elements_set |= {el.tag}

elements_set

{'ATHLETES',
 'CLUBS',
 'ENTRIES',
 'EVENTS',
 'HEATS',
 'MEETS',
 'RELAYPOSITIONS',
 'RELAYS',
 'RESULTS',
 'SESSIONS',
 'SPLITS'}

In [4]:
start, *_, finish = root.find("MEETS").find("MEET").find("SESSIONS").findall("SESSION")
start = {"start": start.attrib["date"]}
finish = {"finish": finish.attrib["date"]}

contest_info = root.find("MEETS").find("MEET").attrib
contest_info["course"] = "25m" if contest_info["course"] == "SCM" else "50m"

del contest_info["timing"]

contest = contest_info | start | finish
contest

{'name': 'European Aquatics Short Course Swimming Championships',
 'city': 'Lublin',
 'nation': 'POL',
 'course': '25m',
 'start': '2025-12-02',
 'finish': '2025-12-07'}

In [5]:
save_path = f"{contest["course"]}/{contest["name"]}"
save_path

'25m/European Aquatics Short Course Swimming Championships'

In [6]:
stroke_dict = {
    "FREE": "freestyle",
    "BACK": "backstroke",
    "FLY": "butterfly",
    "BREAST": "breaststroke",
    "MEDLEY": "IM"
}

gender_dict = {
    "F": "Women's",
    "M": "Men's"
}

round_dict = {
    "PRE": "Heats",
    "SEM": "Semifinals",
    "FIN": "Final",
    "TIM": "Heats",
    "FHT": "Final",
    "QUA": None,
    "SOP": "Swim Off",
    "SOS": "Swim Off",
    "SOQ": "Swim Off",
    "TIMETRIAL": None
}


def get_event(eventid, gender, round, distance, stroke, relaycount, **kwargs):
    if round_dict[round] is None or gender not in ["M", "F"] or int(relaycount) > 1:
        return None

    # return {eventid: [int(distance), stroke_dict[stroke], gender_dict[gender], round_dict[round]]}
    return {eventid: {"event": f"{distance}m {stroke_dict[stroke]} {gender_dict[gender]} {round_dict[round]}", "results": []}}

events = {}
for event in root.iter("EVENT"):
    if event := get_event(**(event.attrib | event.find("SWIMSTYLE").attrib)):
        events |= event

events

{'1': {'event': "400m freestyle Women's Heats", 'results': []},
 '2': {'event': "400m freestyle Men's Heats", 'results': []},
 '3': {'event': "50m butterfly Women's Heats", 'results': []},
 '4': {'event': "50m butterfly Men's Heats", 'results': []},
 '5': {'event': "100m breaststroke Women's Heats", 'results': []},
 '6': {'event': "100m breaststroke Men's Heats", 'results': []},
 '7': {'event': "200m backstroke Women's Heats", 'results': []},
 '8': {'event': "200m backstroke Men's Heats", 'results': []},
 '304': {'event': "50m butterfly Men's Swim Off", 'results': []},
 '101': {'event': "400m freestyle Women's Final", 'results': []},
 '102': {'event': "400m freestyle Men's Final", 'results': []},
 '203': {'event': "50m butterfly Women's Semifinals", 'results': []},
 '204': {'event': "50m butterfly Men's Semifinals", 'results': []},
 '205': {'event': "100m breaststroke Women's Semifinals", 'results': []},
 '206': {'event': "100m breaststroke Men's Semifinals", 'results': []},
 '207': {'

In [7]:
def save_result_swimmer(swimmer: dict):
	os.makedirs(f"{save_path}/swimmers", exist_ok=True)
	
	with open(f"{save_path}/swimmers/{swimmer["fullname"]}.json", "w", encoding="utf-8") as file:
		json.dump(swimmer, file, indent=4)

In [8]:
def get_place(place):
    return {
        "country": place["name"],
        "alpha3": place["code"]
    }


def get_swimmer(athlete, place):
    info = {
        "name": athlete["firstname"],
        "surname": athlete["lastname"],
        "fullname": f"{athlete["firstname"]} {athlete["lastname"]}",
        "sex": gender_dict[athlete["gender"]],
        "birthdate": athlete["birthdate"]
    } | place

    return athlete["athleteid"], info
    return {athlete["athleteid"]: info}


def get_result(swimmer, result: Element):
    result, splits = result.attrib, result.find("SPLITS")

    if "status" in result:
        return {
            "event": events[result["eventid"]]["event"],
            "status": result["status"]
        }

    splits = [split.attrib["swimtime"] for split in splits]
    new_splits = []
    for i in range(len(splits)):
        if i == 0:
            new_splits += [(datetime.datetime.strptime(splits[i], "%H:%M:%S.%f") - datetime.datetime.strptime("00:00:00.00", "%H:%M:%S.%f")).total_seconds()]
        else:
            new_splits += [(datetime.datetime.strptime(splits[i], "%H:%M:%S.%f") - datetime.datetime.strptime(splits[i - 1], "%H:%M:%S.%f")).total_seconds()]

    new_result = {
        "event": events[result["eventid"]]["event"],
        "place": int(result["place"]),
        "heat": int(result["heat"]),
        "lane": int(result["lane"]),
        "reaction_time": float(f"0.{result["reactiontime"][1:]}"),
        "finish_time": result["swimtime"],
        "split": " ".join(map(str, new_splits))
    }

    events[result["eventid"]]["results"] += [dict(list(new_result.items())[1:] + [("name", swimmer["name"]), ("surname", swimmer["surname"]), ("fullname", swimmer["fullname"]), ("country", swimmer["alpha3"])])]

    return new_result


swimmers = {}
for club in root.iter("CLUB"):
    if club.find("ATHLETES").findall("ATHLETE"):
        place = get_place(club.attrib)

        for athlete in club.find("ATHLETES"):
            swimmer_id, swimmer = get_swimmer(athlete.attrib, place)

            results = []
            if athlete.find("RESULTS") is not None:
                for result in athlete.find("RESULTS"):
                    results += [get_result(swimmer, result)]

            swimmer["events"] = results
            swimmers[swimmer_id] = swimmer
            save_result_swimmer(swimmer)


swimmers

{'190919': {'name': 'Klint',
  'surname': 'Hoxha',
  'fullname': 'Klint Hoxha',
  'sex': "Men's",
  'birthdate': '2009-08-27',
  'country': 'Albania',
  'alpha3': 'ALB',
  'events': [{'event': "100m backstroke Men's Heats",
    'place': 44,
    'heat': 1,
    'lane': 8,
    'reaction_time': 0.63,
    'finish_time': '00:00:58.09',
    'split': '13.21 14.35 15.05 15.48'},
   {'event': "200m backstroke Men's Heats",
    'place': 32,
    'heat': 1,
    'lane': 2,
    'reaction_time': 0.58,
    'finish_time': '00:02:10.28',
    'split': '13.96 15.35 16.12 16.67 17.05 17.45 17.12 16.56'},
   {'event': "50m backstroke Men's Heats",
    'place': 47,
    'heat': 1,
    'lane': 0,
    'reaction_time': 0.61,
    'finish_time': '00:00:26.91',
    'split': '13.07 13.84'}]},
 '190923': {'name': 'Grisi',
  'surname': 'Koxhaku',
  'fullname': 'Grisi Koxhaku',
  'sex': "Men's",
  'birthdate': '2006-03-24',
  'country': 'Albania',
  'alpha3': 'ALB',
  'events': [{'event': "100m freestyle Men's Heats",
 

In [9]:
def save_event(event):
	os.makedirs(f"{save_path}/events", exist_ok=True)

	with open(f"{save_path}/events/{event["event"]}.csv", "w", encoding="utf-8", newline="") as file:
		writer = csv.DictWriter(
			file,
			fieldnames=["place", "lane", "heat", "fullname", "name", "surname", "country", "reaction_time", "finish_time", "split"]
		)
		writer.writeheader()
		writer.writerows(event["results"])

In [10]:
for event_id, v in events.items():
    events[event_id]["results"] =  sorted(v["results"], key=lambda x: x["place"])
    save_event(events[event_id])
    
events

{'1': {'event': "400m freestyle Women's Heats",
  'results': [{'place': 1,
    'heat': 4,
    'lane': 4,
    'reaction_time': 0.8,
    'finish_time': '00:03:56.90',
    'split': '13.46 14.58 14.77 14.77 14.83 14.8 14.9 14.91 14.94 14.94 15.03 15.04 15.03 15.11 15.17 14.62',
    'name': 'Isabel',
    'surname': 'Gose',
    'fullname': 'Isabel Gose',
    'country': 'GER'},
   {'place': 2,
    'heat': 4,
    'lane': 5,
    'reaction_time': 0.81,
    'finish_time': '00:04:00.25',
    'split': '13.54 14.57 14.8 14.9 15.13 15.05 15.09 15.17 15.21 15.28 15.32 15.22 15.42 15.34 15.46 14.75',
    'name': 'Simona',
    'surname': 'Quadarella',
    'fullname': 'Simona Quadarella',
    'country': 'ITA'},
   {'place': 3,
    'heat': 3,
    'lane': 4,
    'reaction_time': 0.69,
    'finish_time': '00:04:01.78',
    'split': '13.02 14.47 15.01 15.05 15.45 15.38 15.4 15.39 15.52 15.42 15.38 15.48 15.42 15.3 15.31 14.78',
    'name': 'Freya',
    'surname': 'Colbert',
    'fullname': 'Freya Colbert',
 